# 📘 Semaine 8 — ADC (Analog-to-Digital Converter)

**Cours :** Microcontrôleurs STM32F103C6T6  
**Durée :** 4h30 (1h30 cours + 1h30 atelier + 1h30 homework)  
**Enseignant :** ____________________  
**Étudiant :** ____________________  
**Date :** ____________________

---

## 🎯 Objectifs pédagogiques de la semaine

À la fin de cette semaine, l'étudiant sera capable de :

1. **Expliquer** le principe d'une conversion analogique-numérique (SAR).
2. **Calculer** code ADC ↔ tension avec les formules de conversion.
3. **Décrire** les caractéristiques de l'ADC du STM32F103 (12 bits, 10 canaux).
4. **Configurer** l'ADC1 en mode simple avec HAL + CubeMX.
5. **Lire** un potentiomètre / capteur analogique et envoyer la valeur via UART.

---

## 🗺️ Plan de la semaine

| Partie | Contenu | Durée |
|---|---|---|
| **A — Cours** | Activités 1 à 6 | 1h30 |
| **B — Atelier** | TP8 : potentiomètre + UART | 1h30 |
| **C — Homework** | Exercices 1 à 3 | 1h30 |
| **D — Auto-évaluation** | Checklist finale | 5 min |

---
# 🎓 PARTIE A — COURS INTÉGRÉ (1h30)

## 🔹 Activité 1 — Rappel & mise en contexte (10 min)

### 🔄 Rappel de la semaine 7
- **QCM** : 40 questions — architecture, GPIO, EXTI, TIMER, PWM.
- **Formules** : `F = F_clk / ((PSC+1) × (ARR+1))`, `Duty = CCR / (ARR+1)`.
- **Préparation ADC** : conversion code ↔ tension.

### ✍️ Questions flash (2 min)
1. Quelle est la résolution de l'ADC du STM32F103 ? → ...
2. Que signifie SAR ? → ...
3. Pour V_ref = 3.3 V et 12 bits, quelle est la résolution en mV ? → ...
4. Combien de canaux ADC disponibles sur la Blue Pill ? → ...

### 🎯 Problème à résoudre
Comment faire **lire** par le STM32 une tension analogique (0 à 3.3 V) et la convertir en nombre numérique exploitable ?

---

## 🔹 Activité 2 — Principe de la conversion ADC (20 min)

### 📖 2.1 — Le monde analogique vs numérique

```
Monde analogique                    Monde numérique
                                 
   Tension continue          ──▶      Suite de nombres
   0 V … 3.3 V                      0, 1, 2, …, 4095
                                 
   Infinité de valeurs           Nombre fini de valeurs
```

### 📖 2.2 — Conversion SAR (Successive Approximation Register)

L'ADC du STM32 utilise la méthode **SAR** : comparaisons successives par **dichotomie**.

**Principe pour 12 bits :**
```
1. Comparer V_in à V_ref/2            → bit 11
2. Comparer à V_ref/4 ou 3V_ref/4     → bit 10
3. ... (12 itérations)
4. Résultat final sur 12 bits
```

**Exemple : V_in = 2.5 V, V_ref = 3.3 V**

| Étape | Bit testé | Valeur testée | Comparaison | Résultat |
|---|---|---|---|---|
| 1 | 11 | 1.65 V | V_in > 1.65 → 1 | 1??? |
| 2 | 10 | 2.475 V | V_in > 2.475 → 1 | 11?? |
| 3 | 9 | 2.888 V | V_in < 2.888 → 0 | 110? |
| 4 | 8 | 2.681 V | V_in < → 0 | 1100 |
| ... | ... | ... | ... | ... |

### 📖 2.3 — Temps de conversion

```
T_conv = T_échantillonnage + 12.5 × T_ADC
```
Où `T_ADC` = 1 / F_ADC (14 MHz max sur STM32F103).

**Exemple :** T_échantillonnage = 1.5 cycles, F_ADC = 14 MHz :
```
T_conv = (1.5 + 12.5) / 14e6 = 1 µs
```

### 🐍 Simulation Python — Algorithme SAR (10 min)

Implémentons un ADC SAR 12 bits pour comprendre le fonctionnement interne.

In [ ]:
# ============================================================
# Simulation d'un ADC SAR (Successive Approximation Register)
# ============================================================

def adc_sar(v_in, v_ref=3.3, bits=12, verbose=False):
    """
    Simule une conversion SAR.
    Retourne le code ADC (entier 0..2^bits - 1).
    """
    code = 0
    for i in range(bits - 1, -1, -1):
        # Test : activer temporairement le bit i
        test = code | (1 << i)
        v_test = test / (2 ** bits) * v_ref
        if v_test <= v_in:
            code = test
            if verbose:
                print(f"  Bit {i:>2} : test={v_test:.4f} V  →  V_in ≥  →  bit=1")
        else:
            if verbose:
                print(f"  Bit {i:>2} : test={v_test:.4f} V  →  V_in <  →  bit=0")
    return code

# Test avec V_in = 2.5 V, V_ref = 3.3 V
v_in = 2.5
v_ref = 3.3

print(f"🔍 Conversion SAR pour V_in = {v_in} V (V_ref = {v_ref} V)\n")
code = adc_sar(v_in, v_ref, bits=12, verbose=True)
v_rec = code / 4096 * v_ref

print(f"\n✅ Code final : {code}  (binaire : {code:012b})")
print(f"   Tension reconstruite : {v_rec:.4f} V")
print(f"   Erreur : {(v_rec - v_in)*1000:+.3f} mV")

---

## 🔹 Activité 3 — Formules de conversion (20 min)

### 📖 3.1 — Relations fondamentales

```
                  V_in
Code  =  round( ─────────  × (2^n − 1) )
                  V_ref

                  Code
V_in  =  ─────────────────  × V_ref
              (2^n − 1)

                V_ref
LSB   =  ─────────────    (résolution en volts)
              2^n
```

Où :
- **n** = nombre de bits (12 pour STM32F103)
- **V_ref** = tension de référence (VDDA = 3.3 V typique)
- **Code** = valeur numérique (0 à 4095)

### 📖 3.2 — Table de correspondance (ADC 12 bits, V_ref = 3.3 V)

| Code | Binaire | Tension |
|---|---|---|
| 0 | 0000 0000 0000 | 0 V |
| 1 | 0000 0000 0001 | 0.806 mV |
| 1024 | 0100 0000 0000 | 0.825 V |
| 2048 | 1000 0000 0000 | 1.650 V |
| 3072 | 1100 0000 0000 | 2.475 V |
| 4095 | 1111 1111 1111 | 3.300 V |

**Résolution (1 LSB) :**
```
LSB = 3.3 V / 4096 ≈ 0.806 mV
```

### 🐍 Table de conversion automatique (10 min)

In [ ]:
# ============================================================
# Table de conversion ADC ↔ Tension
# ============================================================

def code_vers_tension(code, v_ref=3.3, bits=12):
    return code / (2 ** bits) * v_ref

def tension_vers_code(v_in, v_ref=3.3, bits=12):
    code = round(v_in / v_ref * (2 ** bits))
    return min(max(code, 0), 2 ** bits - 1)

def afficher_table(v_ref=3.3, bits=12):
    lsb = v_ref / (2 ** bits)
    print(f"📊 ADC {bits} bits — V_ref = {v_ref} V")
    print(f"   Résolution : {lsb*1000:.4f} mV / LSB\n")
    print(f"{'Code':<8}{'Binaire':<18}{'Tension (V)':<15}{'Tension (mV)'}")
    print("-" * 60)
    pas = 2 ** (bits - 3)  # 8 lignes régulières
    for code in [0] + [i * pas for i in range(1, 8)] + [2**bits - 1]:
        v = code_vers_tension(code, v_ref, bits)
        print(f"{code:<8}{code:0{bits}b}    {v:<15.6f}{v*1000:.3f}")

afficher_table()

print("\n🔄 Conversion inverse (tension → code) :\n")
print(f"{'Tension (V)':<15}{'Code':<10}{'Binaire':<18}{'Tension reconstruite'}")
print("-" * 65)
for v in [0, 0.1, 0.5, 1.0, 1.65, 2.0, 2.5, 3.0, 3.3]:
    c = tension_vers_code(v)
    v_r = code_vers_tension(c)
    print(f"{v:<15}{c:<10}{c:012b}    {v_r:.6f} V")

---

## 🔹 Activité 4 — L'ADC du STM32F103C6T6 (20 min)

### 📖 4.1 — Caractéristiques

| Paramètre | Valeur |
|---|---|
| Résolution | **12 bits** (4096 niveaux) |
| Canaux externes | **10** (IN0 à IN9) |
| Canaux internes | 2 (température, V_ref interne) |
| Nombre d'ADC | **2** (ADC1, ADC2) |
| V_ref | VDDA (2.4 – 3.6 V, typ. 3.3 V) |
| F_ADC max | 14 MHz |
| Temps conversion | 1 µs @ 14 MHz (1.5 + 12.5 cycles) |
| Modes | Simple, scan, continu, discontinu |
| Déclencheurs | Software, timer, EXTI |
| Alignement | Droite ou gauche |
| Résolution effective | ~10.5 bits (ENOB) |

### 📖 4.2 — Canaux ADC ↔ broches

| Canal | Broche | Usage |
|---|---|---|
| ADC_IN0 | PA0 | Capteur analogique |
| ADC_IN1 | PA1 | Capteur analogique |
| ADC_IN2 | PA2 | Capteur analogique |
| ADC_IN3 | PA3 | Capteur analogique |
| ADC_IN4 | PA4 | Capteur analogique |
| ADC_IN5 | PA5 | Capteur analogique |
| ADC_IN6 | PA6 | Capteur analogique |
| ADC_IN7 | PA7 | Capteur analogique |
| ADC_IN8 | PB0 | Capteur analogique |
| ADC_IN9 | PB1 | Capteur analogique |
| ADC_IN16 | (interne) | Température |
| ADC_IN17 | (interne) | V_ref interne |

> ⚠️ **Attention** : PA0 et PA1 sont aussi utilisés par TIM2_CH1/CH2 (S6) et EXTI0/EXTI1 (S4).  
> Vérifier les conflits de broches lors de la configuration CubeMX.

### 📖 4.3 — Registres ADC principaux

| Registre | Rôle |
|---|---|
| **SR** | Status Register — EOC (End Of Conversion), AWD |
| **CR1** | Control Register 1 — scan, interrupt enable |
| **CR2** | Control Register 2 — ADON, CONT, CAL, SWSTART |
| **SMPR1/2** | Sample time par canal |
| **SQR1/2/3** | Sequence de conversion |
| **DR** | Data Register — résultat 12 bits |
| **JSQR** | Sequence injected |
| **JDR1..4** | Données injected |

### 📖 4.4 — Bits importants dans CR2

| Bit | Nom | Rôle |
|---|---|---|
| 0 | ADON | ADC ON — active l'ADC |
| 1 | CONT | Mode continu |
| 2 | CAL | Lancer la calibration |
| 3 | RSTCAL | Reset calibration |
| 20 | EXTTRIG | Activer trigger externe |
| 22 | SWSTART | Démarrer une conversion (software) |
| 23 | TSVREFE | Activer capteur température + V_ref |

### 🐍 Simulation Python — Précision ADC (10 min)

Analysons l'impact de la résolution et du bruit.

In [ ]:
# ============================================================
# Analyse de la précision ADC
# ============================================================

def comparer_resolutions(v_in=2.5, v_ref=3.3):
    """Compare différentes résolutions pour une même tension."""
    print(f"📊 Comparaison des résolutions (V_in = {v_in} V, V_ref = {v_ref} V)\n")
    print(f"{'Bits':<8}{'Niveaux':<12}{'LSB (mV)':<12}{'Code':<10}{'V reconstruite':<16}{'Erreur (mV)'}")
    print("-" * 75)
    for bits in [4, 6, 8, 10, 12, 14, 16]:
        niveaux = 2 ** bits
        lsb_mv  = v_ref / niveaux * 1000
        code    = round(v_in / v_ref * niveaux)
        v_rec   = code / niveaux * v_ref
        err_mv  = (v_rec - v_in) * 1000
        print(f"{bits:<8}{niveaux:<12}{lsb_mv:<12.4f}{code:<10}{v_rec:<16.6f}{err_mv:+.3f}")

comparer_resolutions(2.5, 3.3)
print()
comparer_resolutions(1.234, 3.3)

---

## 🔹 Activité 5 — Modes de conversion (15 min)

### 📖 5.1 — Modes principaux

| Mode | Description | Usage |
|---|---|---|
| **Simple** | 1 canal, 1 conversion | Lecture ponctuelle |
| **Continu** | 1 canal, conversions en boucle | Mesure rapide |
| **Scan** | Plusieurs canaux séquentiels | Plusieurs capteurs |
| **Discontinu** | Sous-groupes de scan | Contrôle précis |
| **Injected** | Priorité haute, jusqu'à 4 canaux | Mesure urgente |

### 📖 5.2 — Temps d'échantillonnage

Le temps d'échantillonnage doit être adapté à la **résistance de source**.

| Source | Temps recommandé |
|---|---|
| Faible impédance (< 1 kΩ) | 1.5 cycles |
| Impédance moyenne (1-10 kΩ) | 7.5 cycles |
| Impédance élevée (> 10 kΩ) | 41.5 ou 239.5 cycles |

**Temps total de conversion :**
```
T_conv = (T_sample + 12.5) × T_ADC
```

**Exemple (T_sample = 7.5 cycles, F_ADC = 14 MHz) :**
```
T_conv = (7.5 + 12.5) / 14e6 = 1.43 µs
```

### 📖 5.3 — Déclencheurs (triggers)

| Source | Description |
|---|---|
| **Software** | `SWSTART` ou `HAL_ADC_Start()` |
| **TIM1_CH1..CH3** | Trigger par timer avancé |
| **TIM2_CH2..CH3** | Trigger par timer général |
| **TIM3_CH1..CH4** | Trigger par timer général |
| **TIM4_CH4** | Trigger par timer |
| **EXTI11** | Trigger par ligne EXTI |

> 💡 **Important :** le trigger par timer permet une acquisition **précise et périodique** (utile pour l'audio, les capteurs).

---

## 🔹 Activité 6 — Programmation HAL (15 min)

### 📖 6.1 — Configuration CubeMX

1. *Analog → ADC1 → IN0* (active PA0 en ADC)
2. *Parameter Settings* :
   - Mode : `Independent mode`
   - Clock Prescaler : `/6` → 12 MHz (÷ 2 = OK)
   - Resolution : `12 bits`
   - Data Alignment : `Right`
   - Scan Conversion Mode : `Disabled`
   - Continuous Conversion Mode : `Disabled`
   - Sampling Time : `7.5 Cycles`
3. Générer le code

### 📖 6.2 — Code HAL

```c
HAL_ADC_Start(&hadc1);
HAL_ADC_PollForConversion(&hadc1, 100);   // timeout 100 ms
uint16_t val = HAL_ADC_GetValue(&hadc1);
float tension = val * 3.3f / 4095.0f;
HAL_ADC_Stop(&hadc1);
```

### 📖 6.3 — Calibration

Avant la première conversion, il est recommandé de calibrer :

```c
HAL_ADCEx_Calibration_Start(&hadc1);
```

> 💡 La calibration compense les dérives internes de l'ADC. Durée : ~83 cycles ADC.

In [ ]:
/* ============================================================
   Lecture ADC1 canal 0 (PA0) — version HAL simple
   ============================================================ */

#include "main.h"

ADC_HandleTypeDef hadc1;
UART_HandleTypeDef huart2;

static uint16_t lire_adc(void)
{
    HAL_ADC_Start(&hadc1);
    if (HAL_ADC_PollForConversion(&hadc1, 100) != HAL_OK)
        return 0;
    uint16_t val = HAL_ADC_GetValue(&hadc1);
    HAL_ADC_Stop(&hadc1);
    return val;
}

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_ADC1_Init();
    MX_USART2_UART_Init();

    // Calibration (recommandée)
    HAL_ADCEx_Calibration_Start(&hadc1);

    char buf[64];

    while (1)
    {
        uint16_t val = lire_adc();
        float tension = val * 3.3f / 4095.0f;

        int n = snprintf(buf, sizeof(buf),
                         "ADC = %4u  |  Tension = %.4f V\r\n",
                         val, tension);
        HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);

        HAL_Delay(200);
    }
}

### 📖 6.4 — Version avec interruption

```c
/* Démarrage avec interruption */
HAL_ADC_Start_IT(&hadc1);

/* Callback */
void HAL_ADC_ConvCpltCallback(ADC_HandleTypeDef *hadc)
{
    if (hadc->Instance == ADC1)
    {
        uint16_t val = HAL_ADC_GetValue(hadc);
        // Traitement
    }
}
```

### 📖 6.5 — Version LL (rapide)

```c
LL_ADC_Enable(ADC1);
LL_ADC_REG_StartConversionSWStart(ADC1);
while (!LL_ADC_IsActiveFlag_EOC(ADC1)) {}
uint16_t val = LL_ADC_REG_ReadConversionData12(ADC1);
```

---

## 🔹 Activité 7 — QCM formatif (10 min)

**1. La résolution de l'ADC du STM32F103 est :**  
A. 8 bits  
B. 10 bits  
C. 12 bits  
D. 16 bits

**2. Le nombre de canaux ADC externes disponibles est :**  
A. 4  
B. 8  
C. 10  
D. 16

**3. Pour V_ref = 3.3 V et 12 bits, la résolution en mV est :**  
A. 0.403 mV  
B. 0.806 mV  
C. 1.612 mV  
D. 3.300 mV

**4. Le bit ADON dans CR2 sert à :**  
A. Activer le mode continu  
B. Activer l'ADC  
C. Lancer la calibration  
D. Démarrer une conversion

**5. La méthode de conversion utilisée est :**  
A. Delta-Sigma  
B. Flash  
C. SAR (Successive Approximation)  
D. Double pente

**6. Le flag de fin de conversion est :**  
A. ADON  
B. SWSTART  
C. EOC  
D. CONT

### ✅ Corrigé du QCM formatif

| Q | Réponse | Explication |
|---|---|---|
| 1 | **C — 12 bits** | 4096 niveaux |
| 2 | **C — 10** | IN0 à IN9 |
| 3 | **B — 0.806 mV** | 3.3 / 4096 |
| 4 | **B — Activer l'ADC** | ADON = ADC ON |
| 5 | **C — SAR** | Approximations successives |
| 6 | **C — EOC** | End Of Conversion |

**Mon score : ___ / 6**

---

# 🛠️ PARTIE B — ATELIER / TP (1h30)

## 🧪 TP8 — Lecture potentiomètre + affichage UART

### 🎯 Objectif
Lire une tension analogique (potentiomètre) sur PA0 (ADC_IN0) et envoyer la valeur sur UART2 vers un terminal PC. Afficher aussi la valeur en binaire sur 8 LEDs.

### 📋 Tâches à réaliser (par binôme)

| # | Tâche | Durée | Livrable |
|---|---|---|---|
| 1 | Câbler le potentiomètre sur PA0 (milieu), VCC et GND | 10 min | Photo du montage |
| 2 | Configurer ADC1_IN0 en CubeMX (12 bits, 7.5 cycles) | 15 min | Capture |
| 3 | Configurer USART2 (115200 bauds) | 10 min | Capture |
| 4 | Écrire le code de lecture + envoi | 20 min | Code |
| 5 | Vérifier au multimètre et comparer à la valeur UART | 15 min | Mesure |
| 6 | Bonus : afficher les 8 bits de poids fort sur LEDs | 15 min | Démo |
| 7 | Rédiger le compte-rendu | 5 min | CR |

### ⚙️ Code complet — Lecture ADC + UART + LEDs

In [ ]:
/* ============================================================
   TP8 - Lecture ADC1_IN0 + UART2 + affichage LED
   - PA0 : ADC_IN0 (potentiomètre)
   - PA2 : USART2_TX
   - PA3 : USART2_RX
   - PB0..PB7 : LEDs (8 bits de poids fort)
   ============================================================ */

#include "main.h"

ADC_HandleTypeDef hadc1;
UART_HandleTypeDef huart2;

/* --- Lecture ADC1 avec polling --- */
static uint16_t lire_adc(void)
{
    HAL_ADC_Start(&hadc1);
    if (HAL_ADC_PollForConversion(&hadc1, 100) != HAL_OK)
        return 0;
    uint16_t val = HAL_ADC_GetValue(&hadc1);
    HAL_ADC_Stop(&hadc1);
    return val;
}

/* --- Affichage des 8 bits de poids fort sur PB0..PB7 --- */
static void afficher_leds(uint16_t adc_val)
{
    uint8_t huit_bits = adc_val >> 4;   // 12 bits → 8 bits
    GPIOB->ODR = (GPIOB->ODR & 0xFF00) | huit_bits;
}

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_ADC1_Init();
    MX_USART2_UART_Init();

    // Calibration ADC
    HAL_ADCEx_Calibration_Start(&hadc1);

    char buf[80];

    while (1)
    {
        uint16_t val = lire_adc();
        float tension = val * 3.3f / 4095.0f;

        afficher_leds(val);

        int n = snprintf(buf, sizeof(buf),
                         "ADC=%4u  V=%.4f V  Hex=0x%03X\r\n",
                         val, tension, val);
        HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);

        HAL_Delay(200);
    }
}

### 🔍 Analyse du code

| Élément | Rôle |
|---|---|
| `HAL_ADCEx_Calibration_Start()` | Calibre l'ADC (à faire une fois) |
| `HAL_ADC_Start()` | Démarre une conversion |
| `HAL_ADC_PollForConversion()` | Attend la fin (EOC) |
| `HAL_ADC_GetValue()` | Lit la valeur (DR) |
| `HAL_ADC_Stop()` | Arrête l'ADC |
| `val * 3.3 / 4095` | Conversion en tension |
| `val >> 4` | 12 bits → 8 bits (LEDs) |

### 📖 Configuration CubeMX — ADC1_IN0

```
Mode : Independent mode
Clock Prescaler : /6 → 12 MHz
Resolution : 12 bits
Data Alignment : Right
Scan Conversion Mode : Disabled
Continuous Conversion Mode : Disabled
Discontinuous Conversion Mode : Disabled
Sampling Time Channel 0 : 7.5 Cycles
```

### 📖 Configuration CubeMX — USART2

```
Mode : Asynchronous
Baud Rate : 115200
Word Length : 8 bits
Parity : None
Stop Bits : 1
```

### 🐍 Simulation Python — Potentiomètre & bruit (15 min)

Simulons une lecture ADC avec bruit pour comprendre l'importance du filtrage.

In [ ]:
# ============================================================
# Simulation : lecture ADC avec bruit
# ============================================================

import random
random.seed(42)

def lire_adc_simule(v_reel, v_ref=3.3, bits=12, bruit_mv=5):
    """Simule une lecture ADC avec bruit gaussien."""
    bruit_v = random.gauss(0, bruit_mv / 1000)
    v_mesure = v_reel + bruit_v
    v_mesure = max(0, min(v_ref, v_mesure))
    return int(v_mesure / v_ref * (2 ** bits))

def filtrer_moyenne(valeurs):
    return sum(valeurs) / len(valeurs)

# V_reel = 1.5 V
V_REF = 3.3
V_REEL = 1.5

print(f"📊 Lecture ADC simulée (V_réel = {V_REEL} V, bruit = 5 mV)\n")

N = 20
lectures = [lire_adc_simule(V_REEL, V_REF) for _ in range(N)]

print(f"{'#':<4}{'Code':<8}{'Tension (V)':<14}")
print("-" * 30)
for i, code in enumerate(lectures, 1):
    v = code / 4096 * V_REF
    print(f"{i:<4}{code:<8}{v:<14.4f}")

moyenne = filtrer_moyenne(lectures)
v_moy = moyenne / 4096 * V_REF

print(f"\n📈 Statistiques :")
print(f"  Moyenne  : {moyenne:.2f}  ({v_moy:.4f} V)")
print(f"  Min      : {min(lectures)}  ({min(lectures)/4096*V_REF:.4f} V)")
print(f"  Max      : {max(lectures)}  ({max(lectures)/4096*V_REF:.4f} V)")
print(f"  Écart    : ±{(max(lectures)-min(lectures))/2/4096*V_REF*1000:.2f} mV")
print(f"  Erreur V : {(v_moy - V_REEL)*1000:+.2f} mV")

### 🐍 Comparaison filtres (moyenne glissante, médiane)

In [ ]:
# ============================================================
# Comparaison de filtres sur un signal ADC bruité
# ============================================================

def generer_signal_bruite(n_points, v_reel, bruit_mv=20):
    return [v_reel + random.gauss(0, bruit_mv/1000) for _ in range(n_points)]

def moyenne_glissante(signal, fenetre):
    resultat = []
    for i in range(len(signal)):
        debut = max(0, i - fenetre + 1)
        bloc = signal[debut:i+1]
        resultat.append(sum(bloc) / len(bloc))
    return resultat

def mediane_glissante(signal, fenetre):
    resultat = []
    for i in range(len(signal)):
        debut = max(0, i - fenetre + 1)
        bloc = sorted(signal[debut:i+1])
        mid = len(bloc) // 2
        resultat.append(bloc[mid])
    return resultat

import statistics

V_REEL = 1.5
N      = 30
FEN    = 5

signal = generer_signal_bruite(N, V_REEL, bruit_mv=50)
filtre_moy = moyenne_glissante(signal, FEN)
filtre_med = mediane_glissante(signal, FEN)

print(f"🎯 Signal cible : {V_REEL} V  |  Bruit : ±50 mV\n")
print(f"{'#':<4}{'Brut':<10}{'Moy.glissante':<18}{'Médiane glissante'}")
print("-" * 55)
for i in range(0, N, 3):
    print(f"{i:<4}{signal[i]:<10.4f}{filtre_moy[i]:<18.4f}{filtre_med[i]:.4f}")

print(f"\n📊 Erreur moyenne finale :")
print(f"  Brut              : {(statistics.mean(signal) - V_REEL)*1000:+.3f} mV")
print(f"  Moyenne glissante : {(filtre_moy[-1] - V_REEL)*1000:+.3f} mV")
print(f"  Médiane glissante : {(filtre_med[-1] - V_REEL)*1000:+.3f} mV")

### 📝 Compte-rendu de TP8

**Nom :** __________________  **Prénom :** __________________  **Binôme :** __________________

**1. Configuration CubeMX**
- ADC1 : mode = ... , prescaler = ... , résolution = ...
- Canal : IN...  , sampling time = ...
- USART2 : baudrate = ... , format = ...

**2. Code ajouté dans `main.c`**
```c
// Colle ici ton code
```

**3. Mesures**
| Position potentiomètre | Code ADC | Tension UART (V) | Tension multimètre (V) | Écart (mV) |
|---|---|---|---|---|
| 0 % | ... | ... | ... | ... |
| 25 % | ... | ... | ... | ... |
| 50 % | ... | ... | ... | ... |
| 75 % | ... | ... | ... | ... |
| 100 % | ... | ... | ... | ... |

**4. Analyse des écarts**
- Écart maximal observé : ... mV
- Origines possibles : ...
- Solution proposée : ...

**5. Affichage LED**
- La barre LED progresse-t-elle correctement ? ...
- Y a-t-il du scintillement ? ...
- Filtrage appliqué : ...

**6. Problèmes rencontrés**
- ...

**7. Solutions apportées**
- ...

### 🧪 Exercice bonus — Moyenne glissante en C

Implémenter un filtre à **moyenne glissante** sur N=10 échantillons pour lisser la lecture ADC.

**Cahier des charges :**
- Buffer circulaire de 10 valeurs
- Calculer la moyenne à chaque nouvelle lecture
- Comparer la stabilité avant/après filtrage

In [ ]:
// Solution bonus — Moyenne glissante

#define FILTRE_N  10

typedef struct {
    uint16_t buffer[FILTRE_N];
    uint8_t  index;
    uint32_t somme;
    uint8_t  rempli;
} FiltreMoyenne;

static void filtre_init(FiltreMoyenne *f)
{
    for (int i = 0; i < FILTRE_N; i++) f->buffer[i] = 0;
    f->index = 0;
    f->somme = 0;
    f->rempli = 0;
}

static uint16_t filtre_ajouter(FiltreMoyenne *f, uint16_t val)
{
    f->somme -= f->buffer[f->index];
    f->buffer[f->index] = val;
    f->somme += val;
    f->index = (f->index + 1) % FILTRE_N;
    if (!f->rempli && f->index == 0) f->rempli = 1;
    uint8_t n = f->rempli ? FILTRE_N : f->index;
    return f->somme / n;
}

// Utilisation :
// FiltreMoyenne filtre;
// filtre_init(&filtre);
// uint16_t val_filtree = filtre_ajouter(&filtre, lire_adc());

---

# 🏠 PARTIE C — HOMEWORK (1h30)

## 📚 Exercices à rendre

### 🧩 Exercice 1 — Conversions ADC (30 min)

Pour un ADC 12 bits avec V_ref = **3.3 V**, calculer les codes ADC correspondants.

| # | Tension | Code attendu |
|---|---|---|
| 1 | 0.000 V | ? |
| 2 | 0.100 V | ? |
| 3 | 0.500 V | ? |
| 4 | 1.000 V | ? |
| 5 | 1.234 V | ? |
| 6 | 1.650 V | ? |
| 7 | 2.000 V | ? |
| 8 | 2.500 V | ? |
| 9 | 3.000 V | ? |
| 10 | 3.300 V | ? |

Puis, pour un ADC 10 bits avec V_ref = **5.0 V** :

| # | Tension | Code attendu |
|---|---|---|
| 11 | 0.0 V | ? |
| 12 | 1.0 V | ? |
| 13 | 2.5 V | ? |
| 14 | 4.0 V | ? |
| 15 | 5.0 V | ? |

In [ ]:
# Corrigé Exercice 1

def convertir(v_in, v_ref, bits):
    code = round(v_in / v_ref * (2 ** bits))
    return min(max(code, 0), 2 ** bits - 1)

print("ADC 12 bits, V_ref = 3.3 V\n")
print(f"{'#':<4}{'Tension':<12}{'Code':<10}{'Hex':<10}{'Binaire'}")
print("-" * 55)
for i, v in enumerate([0, 0.1, 0.5, 1.0, 1.234, 1.65, 2.0, 2.5, 3.0, 3.3], 1):
    c = convertir(v, 3.3, 12)
    print(f"{i:<4}{v:<12.4f}{c:<10}0x{c:03X}    {c:012b}")

print("\nADC 10 bits, V_ref = 5.0 V\n")
print(f"{'#':<4}{'Tension':<12}{'Code':<10}{'Hex':<10}{'Binaire'}")
print("-" * 55)
for i, v in enumerate([0.0, 1.0, 2.5, 4.0, 5.0], 11):
    c = convertir(v, 5.0, 10)
    print(f"{i:<4}{v:<12.4f}{c:<10}0x{c:03X}    {c:010b}")

### 🧩 Exercice 2 — Analyse d'un ADC (30 min)

Un ADC 12 bits avec V_ref = 3.3 V renvoie les valeurs suivantes :

```
2048, 2048, 2048, 2047, 2049, 2048, 2048, 2048, 2048, 2048
```

**Questions :**
1. Quelle est la tension mesurée (en V) ?
2. Quelle est la tension réelle si V_in = 1.65 V ?
3. Quelle est l'erreur en mV ?
4. Quelle est la valeur maximale acceptable du bruit pour 1 LSB ?
5. Proposer une méthode de filtrage pour stabiliser la lecture.

In [ ]:
# Corrigé Exercice 2

lectures = [2048, 2048, 2048, 2047, 2049, 2048, 2048, 2048, 2048, 2048]
V_REF = 3.3
BITS  = 12

moy_code = sum(lectures) / len(lectures)
v_moy = moy_code / (2 ** BITS) * V_REF
v_theo = 1.65
lsb_mv = V_REF / (2 ** BITS) * 1000

print(f"1. Tension mesurée  : {v_moy:.6f} V")
print(f"2. Tension théorique: {v_theo:.6f} V  (code idéal = {round(v_theo/V_REF*4096)})")
print(f"3. Erreur           : {(v_moy - v_theo)*1000:+.4f} mV")
print(f"4. LSB              : {lsb_mv:.4f} mV  → bruit max ±{lsb_mv/2:.3f} mV")
print(f"5. Filtrage         : moyenne glissante sur N échantillons")
print(f"                       erreur réduite en 1/√N")

### 🧩 Exercice 3 — Lecture du RM0008 (30 min)

Lire le chapitre 11 (ADC) du RM0008 et répondre :

1. Quelle est la différence entre **regular** et **injected** channels ?
2. Comment activer la **conversion continue** ? Quel bit dans CR2 ?
3. Que fait la **calibration** et combien de temps dure-t-elle ?
4. Quel est le temps d'échantillonnage minimal pour une source < 1 kΩ ?
5. Comment déclencher une conversion par **TIM2_TRGO** ?
6. Que signifie **EOC** et comment l'utiliser en interruption ?

### ✍️ Réponses — Exercice 3

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...

---

## 🧮 Exercice supplémentaire — ADC en mode scan (optionnel)

Configurer ADC1 en mode **scan** pour lire 4 canaux (IN0, IN1, IN2, IN3) en séquence, et calculer la moyenne.

**Indices :**
- Activer `Scan Conversion Mode` dans CubeMX
- Utiliser `HAL_ADC_Start_DMA()` avec un buffer de 4 uint16_t
- Calculer la moyenne sur les 4 valeurs

In [ ]:
// Squelette solution bonus — scan ADC + DMA

#define NB_CANAUX  4

static uint16_t adc_buffer[NB_CANAUX];

/* À compléter :
  1. Configurer ADC1 en mode Scan + DMA circulaire
  2. Lancer : HAL_ADC_Start_DMA(&hadc1, (uint32_t*)adc_buffer, NB_CANAUX)
  3. Calculer la moyenne dans une boucle
  4. Envoyer sur UART
*/

---
# ✅ PARTIE D — AUTO-ÉVALUATION Semaine 8

Coche ce que tu maîtrises.

- [ ] Je connais le principe de la conversion SAR.
- [ ] Je connais les formules : `Code = V_in/V_ref × 2^n` et inverse.
- [ ] Je sais calculer la résolution (LSB) d'un ADC.
- [ ] Je connais les caractéristiques ADC du STM32F103 (12 bits, 10 canaux).
- [ ] Je connais les canaux ADC ↔ broches du STM32F103C6T6.
- [ ] Je connais les registres ADC (CR1, CR2, SR, DR).
- [ ] Je comprends les modes simple / continu / scan.
- [ ] Je sais configurer ADC1 en CubeMX.
- [ ] Je sais utiliser `HAL_ADC_Start()` et `HAL_ADC_PollForConversion()`.
- [ ] J'ai implémenté une lecture potentiomètre + UART.
- [ ] J'ai implémenté une moyenne glissante (filtrage).
- [ ] J'ai comparé UART avec un multimètre.
- [ ] J'ai rédigé mon compte-rendu de TP8.
- [ ] J'ai lu le chapitre 11 du RM0008.

### 📊 Mon score : ___ / 14

| Score | Interprétation |
|---|---|
| 12–14 | ✅ Prêt pour la S9 (ADC + DMA) |
| 8–11 | ⚠️ Revoir les points manquants |
| < 8 | 🔁 Reprendre les activités 2 à 6 |

---
# 📚 RESSOURCES Semaine 8

### Documents officiels
- 📄 **RM0008** — chapitre 11 (Analog-to-digital converter)
- 📄 **Datasheet STM32F103x6** — section 5.3.17 (ADC characteristics)
- 📄 **AN2834** — How to get the best ADC accuracy
- 📄 **AN3116** — ADC modes and their applications

### Outils
- **STM32CubeMX** — Analog → ADC1
- **Multimètre** pour comparer
- **Terminal série** (PuTTY, minicom, screen)

### Vidéos
- *STM32 ADC Tutorial (HAL)* — ControllersTech
- *ADC Sampling Techniques* — YouTube

### Bonnes pratiques
- Toujours calibrer l'ADC avant la première conversion.
- Adapter le **sampling time** à l'impédance source.
- Filtrer les lectures (moyenne, médiane) pour stabiliser.
- Éviter de mélanger ADC et GPIO sur la même broche.
- Vérifier la tension V_ref (VDDA) réelle.

---

### 🔗 Passage à la semaine 9

**Prochaine séance :** ADC + DMA  
- Transfert DMA mémoire ↔ ADC
- Mode circulaire (buffer continu)
- Trigger par timer (acquisition périodique)
- Filtrage avancé : moyenne, médiane
- TP9 : acquisition périodique + détection de seuil

**Préparation :** Lire la section DMA du chapitre 11 du RM0008.

---

**Fin du notebook — Semaine 8** ✨